# Bernstein-Vazirani
The oracle hides a secret string $s$ via $f(x)=s\cdot x \bmod 2$. One query recovers all $n$ bits of $s$ at once.
Measure the input register: the bitstring you read out **is** $s$.
Run twice — an ideal simulator and a noisy one carrying a real IBM device's noise model (offline).

In [ ]:
# Oracle: CX from input qubit i into the target wherever secret bit s_i = 1
from qiskit import QuantumCircuit

def bv_oracle(s: str) -> QuantumCircuit:
    n = len(s)
    oracle = QuantumCircuit(n + 1)
    for i, bit in enumerate(reversed(s)):   # qubit 0 = least-significant bit
        if bit == "1":
            oracle.cx(i, n)
    return oracle

In [ ]:
# Assemble the full Bernstein-Vazirani circuit and draw it
s = "1011"                    # the secret to recover
n = len(s)

qc = QuantumCircuit(n + 1, n)
qc.x(n)                       # target starts in |1>
qc.h(range(n + 1))            # superposition + |-> on target
qc.barrier()
qc.compose(bv_oracle(s), inplace=True)
qc.barrier()
qc.h(range(n))               # interference collapses the input register onto s
qc.measure(range(n), range(n))

qc.draw("mpl")

In [ ]:
# Run on an ideal simulator and a noisy one (FakeSherbrooke = real IBM device noise model, offline)
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

ideal = AerSimulator()
noisy = AerSimulator.from_backend(FakeSherbrooke())

counts_ideal = ideal.run(transpile(qc, ideal), shots=1024).result().get_counts()
counts_noisy = noisy.run(transpile(qc, noisy, optimization_level=3), shots=1024).result().get_counts()

In [ ]:
# The dominant bitstring should equal s; noise spreads weight onto neighbours
from qiskit.visualization import plot_histogram

print("secret s   :", s)
print("ideal peak :", max(counts_ideal, key=counts_ideal.get))
print("noisy peak :", max(counts_noisy, key=counts_noisy.get))
plot_histogram([counts_noisy, counts_ideal], legend=["noisy", "ideal"])